In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [3]:
!pip -q install transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

deberta_checkpoint = "microsoft/deberta-v3-small"
roberta_checkpoint = "roberta-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_checkpoint)
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_checkpoint)

deberta_model = AutoModelForSequenceClassification.from_pretrained(
    deberta_checkpoint,
    num_labels=5
).to(device)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    roberta_checkpoint,
    num_labels=5
).to(device)

deberta_model.eval()
roberta_model.eval()

print(device)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


cuda


In [5]:
import torch.nn.functional as F

prompt = str(train.loc[25, "prompt"])

inputs = deberta_tokenizer(
    prompt,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = deberta_model(**inputs)

probs = F.softmax(outputs.logits, dim=1)[0]

labels = ["A", "B", "C", "D", "E"]

best_idx = torch.argmax(probs).item()

print(labels[best_idx], round(probs[best_idx].item(), 4))
print(dict(zip(labels, probs.cpu().numpy())))

B 0.2517
{'A': np.float16(0.1642), 'B': np.float16(0.2517), 'C': np.float16(0.1732), 'D': np.float16(0.2323), 'E': np.float16(0.1785)}


In [6]:
import torch
import torch.nn.functional as F

labels = ["A", "B", "C", "D", "E"]

prompt = str(train.loc[25, "prompt"])

def get_probability(model, tokenizer):
    inputs = tokenizer(
        prompt,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    return F.softmax(logits, dim=1)[0]

deberta_probs = get_probability(deberta_model, deberta_tokenizer)
roberta_probs = get_probability(roberta_model, roberta_tokenizer)

avg_probs = (deberta_probs + roberta_probs) / 2

best_idx = torch.argmax(avg_probs).item()

print("DeBERTa:", dict(zip(labels, deberta_probs.cpu().numpy())))
print("RoBERTa:", dict(zip(labels, roberta_probs.cpu().numpy())))
print("Average:", dict(zip(labels, avg_probs.cpu().numpy())))

print(labels[best_idx])

DeBERTa: {'A': np.float16(0.1642), 'B': np.float16(0.2517), 'C': np.float16(0.1732), 'D': np.float16(0.2323), 'E': np.float16(0.1785)}
RoBERTa: {'A': np.float32(0.16248015), 'B': np.float32(0.19979788), 'C': np.float32(0.20616774), 'D': np.float32(0.20733932), 'E': np.float32(0.22421491)}
Average: {'A': np.float32(0.16333236), 'B': np.float32(0.22575343), 'C': np.float32(0.18969277), 'D': np.float32(0.21981956), 'E': np.float32(0.20134085)}
B


In [7]:
import torch

labels = ["A", "B", "C", "D", "E"]

weighted_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

best_idx = torch.argmax(weighted_probs).item()

print("Weighted probabilities:")
print(dict(zip(labels, weighted_probs.cpu().numpy())))

print("Top option:", labels[best_idx])

Weighted probabilities:
{'A': np.float32(0.16367325), 'B': np.float32(0.23608683), 'C': np.float32(0.18312718), 'D': np.float32(0.22479945), 'E': np.float32(0.19220343)}
Top option: B


In [8]:
labels = ["A", "B", "C", "D", "E"]

ranking = torch.argsort(weighted_probs, descending=True)

top3 = " ".join([labels[i] for i in ranking[:3]])

print(top3)

B D E


In [9]:
labels = ["A", "B", "C", "D", "E"]

predictions = []

for _, row in test.iterrows():
    prompt = str(row["prompt"])

    def get_probs(model, tokenizer):
        inputs = tokenizer(
            prompt,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits

        return F.softmax(logits, dim=1)[0]

    deberta_probs = get_probs(deberta_model, deberta_tokenizer)
    roberta_probs = get_probs(roberta_model, roberta_tokenizer)

    final_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

    ranking = torch.argsort(final_probs, descending=True)

    top3 = " ".join([labels[i] for i in ranking[:3]])

    predictions.append(top3)

submission = pd.DataFrame({
    "id": test["id"],
    "prediction": predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Prediction rows:", len(submission))

   id prediction
0   1      C D B
1   2      C B D
2   3      D C B
3   4      C A E
4   5      C B D
Prediction rows: 500


In [10]:
labels = ["A", "B", "C", "D", "E"]

def get_deberta_probs(text):
    inputs = deberta_tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = deberta_model(**inputs).logits

    return F.softmax(logits, dim=1)[0]

different_predictions = 0

for i in range(50):
    prompt = str(test.loc[i, "prompt"])

    original_probs = get_deberta_probs(prompt)

    augmented_prompt = "Answer the following multiple-choice question carefully:" + prompt
    augmented_probs = get_deberta_probs(augmented_prompt)

    tta_probs = (original_probs + augmented_probs) / 2

    original_top1 = torch.argmax(original_probs).item()
    tta_top1 = torch.argmax(tta_probs).item()

    if original_top1 != tta_top1:
        different_predictions += 1

print("Different Top-1 predictions:", different_predictions)

Different Top-1 predictions: 7


In [11]:
labels = ["A", "B", "C", "D", "E"]

different_count = 0

def get_probs(model, tokenizer, text):
    inputs = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits

    return F.softmax(logits, dim=1)[0]

for i in range(100):
    prompt = str(test.loc[i, "prompt"])

    deberta_probs = get_probs(
        deberta_model,
        deberta_tokenizer,
        prompt
    )

    roberta_probs = get_probs(
        roberta_model,
        roberta_tokenizer,
        prompt
    )

    ensemble_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

    deberta_top1 = torch.argmax(deberta_probs).item()
    ensemble_top1 = torch.argmax(ensemble_probs).item()

    if deberta_top1 != ensemble_top1:
        different_count += 1

print("Different Top-1 predictions:", different_count)

Different Top-1 predictions: 6


In [12]:
positive_gain = 0

for i in range(100):
    prompt = str(test.loc[i, "prompt"])

    deberta_probs = get_probs(
        deberta_model,
        deberta_tokenizer,
        prompt
    )

    roberta_probs = get_probs(
        roberta_model,
        roberta_tokenizer,
        prompt
    )

    ensemble_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

    deberta_confidence = torch.max(deberta_probs).item()
    ensemble_confidence = torch.max(ensemble_probs).item()

    confidence_gain = ensemble_confidence - deberta_confidence

    if confidence_gain > 0:
        positive_gain += 1

print("Positive confidence gain rows:", positive_gain)

Positive confidence gain rows: 0


In [13]:
labels = ["A", "B", "C", "D", "E"]

changed_top3 = 0

for i in range(100):
    prompt = str(test.loc[i, "prompt"])

    deberta_probs = get_probs(
        deberta_model,
        deberta_tokenizer,
        prompt
    )

    roberta_probs = get_probs(
        roberta_model,
        roberta_tokenizer,
        prompt
    )

    ensemble_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

    deberta_top3 = [
        labels[j]
        for j in torch.argsort(deberta_probs, descending=True)[:3]
    ]

    ensemble_top3 = [
        labels[j]
        for j in torch.argsort(ensemble_probs, descending=True)[:3]
    ]

    if deberta_top3 != ensemble_top3:
        changed_top3 += 1

print("Rows with changed Top-3 ranking:", changed_top3)

Rows with changed Top-3 ranking: 34


In [14]:
labels = ["A", "B", "C", "D", "E"]

validation = train.iloc[:100]

map_score = 0

for i, row in validation.iterrows():
    prompt = str(row["prompt"])

    deberta_probs = get_probs(
        deberta_model,
        deberta_tokenizer,
        prompt
    )

    roberta_probs = get_probs(
        roberta_model,
        roberta_tokenizer,
        prompt
    )

    ensemble_probs = (0.7 * deberta_probs) + (0.3 * roberta_probs)

    top3 = [
        labels[j]
        for j in torch.argsort(ensemble_probs, descending=True)[:3]
    ]

    true_label = row["answer"]

    if true_label in top3:
        rank = top3.index(true_label) + 1
        map_score += 1 / rank

map_score /= len(validation)

print(round(map_score, 4))

0.3817
